<a href="https://colab.research.google.com/github/AmartyaInsan/Data-Analysis/blob/main/Portfolio_Sales_Inventory_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PT Aneka Sport — Sales & Inventory Data Pipeline

End-to-end batch pipeline for PT Aneka Sport's sales, customer, product and
inventory data. Raw pipe-delimited exports are ingested, cleaned, modeled into
a star schema, and aggregated into business-ready data marts.

**Flow:** raw `.txt` exports → Bronze (raw) → Silver (cleaned/typed) → Gold
(star schema: dimensions + facts) → Data Mart (CSV, for BI tools) + KPI
queries.

**Team:** Kelompok 4, NDDE2A — Winda Mailindra, Zaora Zulmianah Anah,
Fathan Nuha Octovan, Amartya Maulana Insan.

**Stack:** PySpark (batch ETL, window functions for running balances and FIFO
aging), Parquet (Silver/Gold storage), Spark SQL (KPI layer).


## Setup

In [1]:
from pathlib import Path
import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType

spark = SparkSession.builder.appName("aneka-sport-pipeline").getOrCreate()


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# All paths live here instead of being redefined inline throughout the
# notebook (the original had gold_dir/silver_dir redeclared 3-4 times each).

ROOT = "/content/drive/MyDrive/Belajar_Big_Data"

PATHS = {
    "raw": f"{ROOT}/dataset/portfolio/",
    "silver": f"{ROOT}/dataset/portfolio/silver_parquet",
    "gold": f"{ROOT}/gold",
    "datamart": f"{ROOT}/datamart",
    "logs": f"{ROOT}/_logs",
}

for p in ["silver", "gold", "datamart", "logs"]:
    Path(PATHS[p]).mkdir(parents=True, exist_ok=True)


## Pipeline logging

Defined early since every stage below reports into it.

In [4]:
import csv

LOG_FILE = f"{PATHS['logs']}/pipeline_log.csv"


def log_step(step, status, rows=None):
    """Append one line to the pipeline run log."""
    file_exists = Path(LOG_FILE).exists()
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["timestamp", "step", "status", "rows"])
        writer.writerow([datetime.datetime.now(), step, status, rows])


## 1. Bronze Layer — Raw Ingestion

Source files are `||`-delimited `.txt` exports with no schema. We read them as
plain text, split on the header row to get column names, then split every
data row into typed-as-string columns.

In [5]:
SOURCE_FILES = [
    "tbl_master_salesman.txt",
    "tbl_master_territory.txt",
    "tbl_master_product.txt",
    "tbl_master_location.txt",
    "tbl_master_customer.txt",
    "tbl_invoice_header.txt",
    "tbl_invoice_detail.txt",
    "tbl_inventory.txt",
    "tbl_inventory_beginning_balance.txt",
]


def load_bronze_table(path):
    """Parse a `||`-delimited raw export into a DataFrame with string columns."""
    raw = spark.read.text(path)
    header = raw.first()["value"]
    columns = [c.strip() for c in header.split("||")]

    data = raw.filter(F.col("value") != header)
    split_col = F.split(F.col("value"), r"\|\|")

    return data.select(
        *[F.trim(split_col.getItem(i)).alias(columns[i]) for i in range(len(columns))]
    )


bronze_dfs = {}
for filename in SOURCE_FILES:
    table_name = filename.replace(".txt", "")
    bronze_dfs[table_name] = load_bronze_table(f"{PATHS['raw']}{filename}")
    print(f"{table_name:<35} {bronze_dfs[table_name].count():>8,} rows")

log_step("bronze_ingestion", "SUCCESS", sum(df.count() for df in bronze_dfs.values()))


tbl_master_salesman                       87 rows
tbl_master_territory                      15 rows
tbl_master_product                    37,692 rows
tbl_master_location                      277 rows
tbl_master_customer                    1,138 rows
tbl_invoice_header                     8,061 rows
tbl_invoice_detail                    59,234 rows
tbl_inventory                          6,293 rows
tbl_inventory_beginning_balance       15,757 rows


## 2. Data Quality Profiling

For each table: row count, duplicate rows, and null/empty counts per column.

The original version of this check ran `.count()` inside a Python loop once
per column — for a 20-column table that's 40+ separate Spark jobs just to
profile one table. Rewritten below to compute null and empty counts for
**every column in a single aggregation pass**, so profiling a table costs one
job instead of dozens.

In [6]:
def profile_table(df):
    """Row count, duplicate count, and per-column null/empty counts in one job."""
    row_count = df.count()
    duplicate_rows = row_count - df.dropDuplicates().count()

    exprs = []
    for column, dtype in df.dtypes:
        exprs.append(F.sum(F.col(column).isNull().cast("int")).alias(f"{column}__nulls"))
        if dtype == "string":
            exprs.append(F.sum((F.trim(F.col(column)) == "").cast("int")).alias(f"{column}__empty"))
        else:
            exprs.append(F.lit(0).alias(f"{column}__empty"))

    stats = df.agg(*exprs).first().asDict()
    column_report = [
        (c, stats[f"{c}__nulls"], stats[f"{c}__empty"]) for c, _ in df.dtypes
    ]

    return row_count, duplicate_rows, column_report


summary_rows = []
for table_name, df in bronze_dfs.items():
    row_count, duplicate_rows, column_report = profile_table(df)
    summary_rows.append({
        "Table": table_name,
        "Rows": row_count,
        "Columns": len(df.columns),
        "Duplicate Rows": duplicate_rows,
    })

    print(f"\n{table_name}  ({row_count:,} rows, {duplicate_rows} duplicate rows)")
    spark.createDataFrame(column_report, ["column", "null_count", "empty_count"]).show(truncate=False)

spark.createDataFrame(summary_rows).show(truncate=False)



tbl_master_salesman  (87 rows, 0 duplicate rows)
+--------+----------+-----------+
|column  |null_count|empty_count|
+--------+----------+-----------+
|CODESLSP|0         |0          |
|NAMEEMPL|0         |0          |
+--------+----------+-----------+


tbl_master_territory  (15 rows, 0 duplicate rows)
+-------------+----------+-----------+
|column       |null_count|empty_count|
+-------------+----------+-----------+
|Kode_Teritory|0         |0          |
|Description  |0         |0          |
+-------------+----------+-----------+


tbl_master_product  (37,692 rows, 0 duplicate rows)
+-----------+----------+-----------+
|column     |null_count|empty_count|
+-----------+----------+-----------+
|ITEMNO     |0         |0          |
|DESC       |0         |0          |
|INACTIVE   |0         |0          |
|CATEGORY   |0         |0          |
|CNTLACCT   |0         |0          |
|STOCKUNIT  |0         |0          |
|DEFPRICLST |0         |0          |
|BRAND      |0         |0          |

#### Insight

| Table | Status |
| --- | --- |
| Salesman, Territory, Inventory | Clean |
| Product, Location, Customer | Optional fields empty — expected |
| Invoice Header | Clean |
| Invoice Detail | Duplicate rows present, investigated below |
| Inventory Beginning Balance | Duplicate rows present, investigated below |


## 3. Primary Key Validation

In [7]:
PRIMARY_KEYS = {
    "tbl_master_salesman": ["CODESLSP"],
    "tbl_master_territory": ["Kode_Teritory"],
    "tbl_master_product": ["ITEMNO"],
    "tbl_master_location": ["LOCATION"],
    "tbl_master_customer": ["IDCUST"],
    "tbl_invoice_header": ["INVUNIQ"],
    "tbl_invoice_detail": ["INVUNIQ", "ITEM"],
    "tbl_inventory": ["DOCNUM", "LOCATION", "ITEMNO", "ENTRYSEQ", "LINENO"],
    "tbl_inventory_beginning_balance": ["ITEMNO", "LOCATION"],
}


def validate_primary_key(df, keys):
    total_rows = df.count()
    unique_rows = df.select(*keys).distinct().count()
    duplicate_keys = df.groupBy(*keys).count().filter(F.col("count") > 1)
    return total_rows, unique_rows, duplicate_keys


pk_report = []
for table_name, keys in PRIMARY_KEYS.items():
    total_rows, unique_rows, duplicate_keys = validate_primary_key(bronze_dfs[table_name], keys)
    duplicate_count = duplicate_keys.count()

    pk_report.append({
        "Table": table_name,
        "Primary Key": ", ".join(keys),
        "Rows": total_rows,
        "Unique Keys": unique_rows,
        "Duplicate Keys": duplicate_count,
        "Status": "PASS" if total_rows == unique_rows else "FAIL",
    })

    if duplicate_count > 0:
        print(f"{table_name}: {duplicate_count} duplicate key groups")
        duplicate_keys.show(5)

spark.createDataFrame(pk_report).show(truncate=False)


tbl_invoice_detail: 314 duplicate key groups
+-------+---------+-----+
|INVUNIQ|     ITEM|count|
+-------+---------+-----+
|1255489|P20205-42|    5|
|1256220|100654-43|    2|
|1255490|P20204-39|    4|
|1260545|P20159-39|    3|
|1259361|400554-41|    2|
+-------+---------+-----+
only showing top 5 rows
tbl_inventory_beginning_balance: 1997 duplicate key groups
+-------------+--------+-----+
|       ITEMNO|LOCATION|count|
+-------------+--------+-----+
| BDVCT15508-M|  CSW007|    4|
|BDVCT82501-XL|   CSWEB|    4|
|K1GR150305-41|     DMA|    4|
|J1GC160904-43|  CSS001|    2|
|K1GA160303-41|  CSW008|    2|
+-------------+--------+-----+
only showing top 5 rows
+--------------+------------------------------------------+-----+------+-------------------------------+-----------+
|Duplicate Keys|Primary Key                               |Rows |Status|Table                          |Unique Keys|
+--------------+------------------------------------------+-----+------+-----------------------------

#### Insight

Sampling the duplicate keys in `tbl_invoice_detail` shows fully identical
rows across every column — not a key-collision, an exact duplicate record
from the source system. Safe to resolve with `dropDuplicates()` in Silver.

## 4. Silver Layer — Cleaning & Standardization

In [8]:
def trim_all_strings(df):
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(c, F.trim(F.col(c)))
    return df


def empty_to_null(df):
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c)))
    return df


def cast_columns(df, mapping):
    for c, dtype in mapping.items():
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast(dtype))
    return df


def cast_dates(df, columns, fmt):
    for c in columns:
        if c in df.columns:
            df = df.withColumn(c, F.to_date(F.col(c), fmt))
    return df


### Deduplicate `tbl_invoice_detail` and `tbl_inventory_beginning_balance`

In [9]:
silver_dfs = bronze_dfs.copy()

for table_name in ["tbl_invoice_detail", "tbl_inventory_beginning_balance"]:
    before = silver_dfs[table_name].count()
    silver_dfs[table_name] = silver_dfs[table_name].dropDuplicates()
    after = silver_dfs[table_name].count()
    print(f"{table_name}: {before:,} -> {after:,} rows ({before - after} exact duplicates removed)")


tbl_invoice_detail: 59,234 -> 58,816 rows (418 exact duplicates removed)
tbl_inventory_beginning_balance: 15,757 -> 11,766 rows (3991 exact duplicates removed)


### Type casting and date parsing

Each table's numeric columns, date columns, and date format live in one
config dict instead of nine separate copy-pasted blocks — new tables just
need an entry here, not a new block of repeated `trim_all_strings` /
`empty_to_null` / `cast_columns` calls.

In [10]:
SILVER_SCHEMA = {
    "tbl_master_product": {
        "numeric": {
            "INACTIVE": IntegerType(), "SEASONCRN": IntegerType(), "SEASONEND": IntegerType(),
            "UnitPrice": DoubleType(), "RBR-Price": DoubleType(), "RTM-Price": DoubleType(),
        },
    },
    "tbl_master_customer": {},
    "tbl_master_salesman": {},
    "tbl_master_location": {},
    "tbl_master_territory": {},
    "tbl_invoice_header": {
        "numeric": {"INVDISCPER": DoubleType(), "INVNETWTX": DoubleType()},
        "dates": ["ORDDATE", "SHIPDATE", "INVDATE"],
        "date_format": "yyyy/MM/dd HH:mm:ss.SSSSSSSSS",
    },
    "tbl_invoice_detail": {
        "numeric": {
            "QTYSHIPPED": DoubleType(), "PRIBASPRC": DoubleType(), "EXTICOST": DoubleType(),
            "EXTINVMISC": DoubleType(), "UNITPRICE": DoubleType(), "INVDISC": DoubleType(),
            "DISCPER": DoubleType(),
        },
    },
    "tbl_inventory": {
        "numeric": {
            "FISCYEAR": IntegerType(), "FISCPERIOD": IntegerType(), "DAYENDSEQ": IntegerType(),
            "ENTRYSEQ": IntegerType(), "LINENO": IntegerType(), "AUDTTIME": IntegerType(),
            "TRANSTYPE": IntegerType(), "QUANTITY": DoubleType(), "CONVERSION": DoubleType(),
            "TRANSCOST": DoubleType(), "STKQTY": DoubleType(), "OPTAMT": DoubleType(),
            "TOTALCOST": DoubleType(), "RECENTCOST": DoubleType(), "COST1": DoubleType(),
            "COST2": DoubleType(), "LASTCOST": DoubleType(), "STDCOST": DoubleType(),
            "COSTCONV": DoubleType(), "TOTALQTY": DoubleType(), "PRICEDECS": DoubleType(),
            "BASEPRICE": DoubleType(), "BASECONV": DoubleType(), "DETAILNUM": IntegerType(),
            "COMPNUM": IntegerType(),
        },
        "dates": ["TRANSDATE", "AUDTDATE", "DATEBUS"],
        "date_format": "yyyyMMdd",
    },
    "tbl_inventory_beginning_balance": {
        "numeric": {
            "QTYONHAND": DoubleType(), "Unit Cost": DoubleType(), "TOTALCOST": DoubleType(),
            "Retail Price": DoubleType(), "SEASONCRN": IntegerType(),
        },
    },
}

cleaned = {}
for table_name, df in silver_dfs.items():
    df = trim_all_strings(df)
    df = empty_to_null(df)

    schema = SILVER_SCHEMA.get(table_name, {})
    if schema.get("dates"):
        df = cast_dates(df, schema["dates"], schema["date_format"])
    if schema.get("numeric"):
        df = cast_columns(df, schema["numeric"])

    cleaned[table_name] = df

silver_dfs = cleaned
print("Silver Layer transformation complete for", len(silver_dfs), "tables.")


Silver Layer transformation complete for 9 tables.


Trims whitespace, converts empty strings to `NULL`, casts numeric
columns, and parses date columns to Spark `date` — applied consistently
across every table via the config above.

### Write Silver Parquet

In [11]:
for table_name, df in silver_dfs.items():
    df.write.mode("overwrite").parquet(f"{PATHS['silver']}/{table_name}")

log_step("silver_layer", "SUCCESS", sum(df.count() for df in silver_dfs.values()))

silver_parquet = {
    folder.name: spark.read.parquet(str(folder))
    for folder in Path(PATHS["silver"]).iterdir() if folder.is_dir()
}
print(list(silver_parquet.keys()))


['tbl_master_salesman', 'tbl_master_territory', 'tbl_master_product', 'tbl_master_location', 'tbl_master_customer', 'tbl_invoice_header', 'tbl_invoice_detail', 'tbl_inventory', 'tbl_inventory_beginning_balance']


## 5. Gold Layer — Star Schema

**Dimensions:** `dim_product`, `dim_customer`, `dim_salesman`, `dim_location`,
`dim_territory`
**Facts:** `fact_sales`, `fact_inventory_daily_balance`, `fact_inventory_aging`

### Dimension tables

Each dimension is a source table, a `{source_column: target_column}` rename
map, and a dedup key — built through one function instead of five nearly
identical select/alias/dropDuplicates/write blocks.

In [12]:
DIMENSION_CONFIG = {
    "dim_product": {
        "source": "tbl_master_product",
        "key": "product_id",
        "columns": {
            "ITEMNO": "product_id", "DESC": "product_name", "CATEGORY": "CATEGORY",
            "BRAND": "BRAND", "PRODUCT": "PRODUCT", "PRODCATG": "PRODCATG",
            "SPORTCAT": "SPORTCAT", "SEASONCRN": "SEASONCRN", "SEASONEND": "SEASONEND",
            "COLOUR": "COLOUR", "SIZE": "SIZE", "UnitPrice": "UnitPrice",
        },
    },
    "dim_customer": {
        "source": "tbl_master_customer",
        "key": "customer_id",
        "columns": {
            "IDCUST": "customer_id", "NAMECUST": "customer_name", "NAMECITY": "city",
            "CODESTTE": "state", "CODECTRY": "country", "CODETERR": "territory_id",
            "CODESLSP1": "salesman_id",
        },
    },
    "dim_salesman": {
        "source": "tbl_master_salesman",
        "key": "salesman_id",
        "columns": {"CODESLSP": "salesman_id", "NAMEEMPL": "salesman_name"},
    },
    "dim_location": {
        "source": "tbl_master_location",
        "key": "location_id",
        "columns": {
            "LOCATION": "location_id", "DESC": "location_name", "CITY": "CITY",
            "STATE": "STATE", "COUNTRY": "COUNTRY",
        },
    },
    "dim_territory": {
        "source": "tbl_master_territory",
        "key": "territory_id",
        "columns": {"Kode_Teritory": "territory_id", "Description": "territory_name"},
    },
}


def build_dimension(source_df, column_map, key):
    selected = source_df.select(*[F.col(src).alias(dst) for src, dst in column_map.items()])
    return selected.dropDuplicates([key])


gold_tables = {}
for dim_name, cfg in DIMENSION_CONFIG.items():
    dim_df = build_dimension(silver_parquet[cfg["source"]], cfg["columns"], cfg["key"])
    gold_tables[dim_name] = dim_df
    dim_df.write.mode("overwrite").parquet(f"{PATHS['gold']}/{dim_name}")
    print(f"{dim_name:<15} {dim_df.count():>6,} rows")

log_step("gold_dimensions", "SUCCESS", sum(df.count() for df in gold_tables.values()))


dim_product     37,692 rows
dim_customer     1,138 rows
dim_salesman        87 rows
dim_location       277 rows
dim_territory       15 rows


### Fact: Sales

In [13]:
invoice_header = silver_parquet["tbl_invoice_header"]
invoice_detail = silver_parquet["tbl_invoice_detail"]

fact_sales = (
    invoice_detail.alias("d")
    .join(invoice_header.alias("h"), F.col("d.INVUNIQ") == F.col("h.INVUNIQ"), "inner")
    .select(
        F.col("h.INVUNIQ").alias("invoice_id"),
        F.col("h.INVNUMBER").alias("invoice_number"),
        F.col("h.INVDATE").alias("invoice_date"),
        F.col("h.CUSTOMER").alias("customer_id"),
        F.col("h.LOCATION").alias("location_id"),
        F.col("h.SALESPER1").alias("salesman_id"),
        F.col("d.ITEM").alias("product_id"),
        F.col("d.QTYSHIPPED").alias("quantity"),
        F.col("d.UNITPRICE").alias("unit_price"),
        F.col("d.EXTICOST").alias("cost"),
        F.col("d.INVDISC").alias("discount"),
    )
    .withColumn("sales_amount", F.col("quantity") * F.col("unit_price"))
)

fact_sales.write.mode("overwrite").parquet(f"{PATHS['gold']}/fact_sales")
gold_tables["fact_sales"] = fact_sales

print(f"fact_sales: {fact_sales.count():,} rows")
fact_sales.printSchema()
log_step("fact_sales", "SUCCESS", fact_sales.count())


fact_sales: 58,816 rows
root
 |-- invoice_id: string (nullable = true)
 |-- invoice_number: string (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- location_id: string (nullable = true)
 |-- salesman_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- sales_amount: double (nullable = true)



### Fact: Inventory Daily Balance

Running stock level per item/location/day: start from the beginning balance,
then accumulate daily net movement with a cumulative window sum.

*(The source notebook built this table twice — once with `season`, once
without — both writing to the same output path, so whichever cell ran last
silently won. Kept the version with `season` since it's the one actually
used downstream for seasonal reporting.)*

In [14]:
def build_daily_inventory_balance(trans_df, begin_df):
    min_date = trans_df.select(F.min("TRANSDATE")).first()[0]
    max_date = trans_df.select(F.max("TRANSDATE")).first()[0]

    date_spine = (
        spark.range((max_date - min_date).days + 1)
        .withColumn("date", F.date_add(F.lit(min_date), F.col("id").cast("int")))
        .select("date")
    )

    items_locations = begin_df.select(
        F.col("ITEMNO").alias("item_no"),
        F.col("LOCATION").alias("location"),
        F.col("SEASONCRN").alias("season"),
    ).distinct()

    daily_spine = items_locations.crossJoin(date_spine)

    daily_movement = trans_df.groupBy(
        F.col("ITEMNO").alias("item_no"),
        F.col("LOCATION").alias("location"),
        F.col("TRANSDATE").alias("date"),
    ).agg(
        F.sum("QUANTITY").alias("net_qty"),
        F.sum("TRANSCOST").alias("net_value"),
    )

    opening_balance = begin_df.select(
        F.col("ITEMNO").alias("item_no"),
        F.col("LOCATION").alias("location"),
        F.col("SEASONCRN").alias("season"),
        F.lit(min_date).alias("date"),
        F.col("QTYONHAND").alias("base_qty"),
        F.col("TOTALCOST").alias("base_value"),
    )

    combined = (
        daily_spine
        .join(opening_balance, on=["item_no", "location", "season", "date"], how="left")
        .join(daily_movement, on=["item_no", "location", "date"], how="left")
        .fillna(0, subset=["base_qty", "base_value", "net_qty", "net_value"])
    )

    running_total = (
        Window.partitionBy("item_no", "location", "season")
        .orderBy("date")
        .rowsBetween(Window.unboundedPreceding, 0)
    )

    return (
        combined
        .withColumn("closing_qty", F.sum(F.col("base_qty") + F.col("net_qty")).over(running_total))
        .withColumn("closing_value", F.sum(F.col("base_value") + F.col("net_value")).over(running_total))
        .select("date", "item_no", "location", "season", "closing_qty", "closing_value")
    )


daily_balance = build_daily_inventory_balance(
    silver_parquet["tbl_inventory"],
    silver_parquet["tbl_inventory_beginning_balance"],
)

daily_balance.write.mode("overwrite").partitionBy("date").parquet(
    f"{PATHS['gold']}/fact_inventory_daily_balance/"
)
gold_tables["fact_inventory_daily_balance"] = daily_balance

print(f"fact_inventory_daily_balance: {daily_balance.count():,} rows")
log_step("fact_inventory_daily_balance", "SUCCESS", daily_balance.count())


fact_inventory_daily_balance: 11,766 rows


### Fact: Inventory Aging (FIFO)

Splits current stock into age buckets (`<=1 year`, `2 years`, `>=3 years`)
by matching issues against receipts in FIFO order.

**Fix:** the source notebook set `analysis_date = datetime.date.today()`.
Because that's the *run* date rather than a date grounded in the data, the
aging buckets shift on every rerun — and if today's wall-clock date is far
ahead of the dataset's actual transaction dates, nearly everything lands in
`>=3 years` regardless of true age (this is what the "why is everything
>=3 years" debugging cell in the original notebook was chasing). Defaulting
`reference_date` to the **latest transaction date in the data** makes the
result reproducible and correct; it can still be overridden for a specific
as-of-date analysis.

In [15]:
def build_inventory_aging(trans_df, product_df, reference_date=None):
    if reference_date is None:
        reference_date = trans_df.select(F.max("TRANSDATE")).first()[0]

    season_map = product_df.select(
        F.col("ITEMNO").alias("item_no"), F.col("SEASONCRN").alias("season")
    ).distinct()

    movements = (
        trans_df
        .join(season_map, F.col("ITEMNO") == F.col("item_no"), "left")
        .filter(F.col("TRANSDATE") <= F.lit(reference_date))
    )

    receipts = movements.filter(F.col("QUANTITY") > 0).select(
        F.col("ITEMNO").alias("item_no"),
        F.col("LOCATION").alias("location"),
        "season",
        F.col("TRANSDATE").alias("receipt_date"),
        F.col("QUANTITY").alias("receipt_qty"),
        F.col("TRANSCOST").alias("receipt_value"),
    )

    issues = (
        movements.filter(F.col("QUANTITY") <= 0)
        .withColumn("issue_qty", F.abs(F.col("QUANTITY")))
        .groupBy(F.col("ITEMNO").alias("item_no"), F.col("LOCATION").alias("location"), "season")
        .agg(F.sum("issue_qty").alias("total_issue_qty"))
    )

    fifo_window = Window.partitionBy("item_no", "location", "season").orderBy("receipt_date")
    receipts_cum = (
        receipts
        .withColumn("cum_receipt", F.sum("receipt_qty").over(fifo_window))
        .withColumn("prev_cum_receipt", F.coalesce(F.lag("cum_receipt").over(fifo_window), F.lit(0)))
    )

    remaining = (
        receipts_cum
        .join(issues, on=["item_no", "location", "season"], how="left")
        .fillna(0, subset=["total_issue_qty"])
        .withColumn(
            "remaining_qty",
            F.greatest(
                F.lit(0),
                F.least(
                    F.col("cum_receipt"),
                    F.col("cum_receipt") - F.col("total_issue_qty") + F.col("receipt_qty"),
                ) - F.col("prev_cum_receipt"),
            ),
        )
        .filter(F.col("remaining_qty") > 0)
        .withColumn("remaining_value", F.col("remaining_qty") * (F.col("receipt_value") / F.col("receipt_qty")))
        .withColumn("age_days", F.datediff(F.lit(reference_date), F.col("receipt_date")))
        .withColumn(
            "age_bucket",
            F.when(F.col("age_days") <= 365, "<=1 year")
             .when(F.col("age_days") <= 730, "2 years")
             .otherwise(">=3 years"),
        )
    )

    return (
        remaining
        .groupBy("item_no", "location", "season", "age_bucket")
        .agg(F.sum("remaining_qty").alias("total_qty"), F.sum("remaining_value").alias("total_value"))
        .withColumn("analysis_date", F.lit(reference_date))
    )


aging_fact = build_inventory_aging(silver_parquet["tbl_inventory"], silver_parquet["tbl_master_product"])

aging_fact.write.mode("overwrite").partitionBy("analysis_date").parquet(
    f"{PATHS['gold']}/fact_inventory_aging/"
)
gold_tables["fact_inventory_aging"] = aging_fact

aging_fact.groupBy("age_bucket").agg(
    F.sum("total_qty").alias("stock_qty"), F.sum("total_value").alias("stock_value")
).show()

log_step("fact_inventory_aging", "SUCCESS", aging_fact.count())


+----------+---------+--------------------+
|age_bucket|stock_qty|         stock_value|
+----------+---------+--------------------+
|  <=1 year|   9345.0|1.3714881832666695E9|
+----------+---------+--------------------+



## 6. Business KPIs (Spark SQL)

Sales achievement by period, product/brand, salesperson, territory, product
category, and city — all answerable directly against the Gold tables.

In [16]:
for name in ["fact_sales", "dim_product", "dim_customer", "dim_salesman", "dim_territory"]:
    spark.read.parquet(f"{PATHS['gold']}/{name}").createOrReplaceTempView(name)

ref_date = spark.sql("SELECT MAX(invoice_date) AS max_date FROM fact_sales").first()["max_date"]
print(f"KPIs calculated relative to: {ref_date}")


KPIs calculated relative to: 2017-02-28


In [17]:
# MTD / YTD sales, relative to the latest invoice date in the data
spark.sql(f"""
    SELECT
        SUM(CASE WHEN month(invoice_date) = month(DATE'{ref_date}')
                  AND year(invoice_date) = year(DATE'{ref_date}') THEN sales_amount ELSE 0 END) AS mtd_sales,
        SUM(CASE WHEN year(invoice_date) = year(DATE'{ref_date}') THEN sales_amount ELSE 0 END) AS ytd_sales
    FROM fact_sales
""").show()


+---------------+---------------+
|      mtd_sales|      ytd_sales|
+---------------+---------------+
|5.2858813095E10|1.0062725346E11|
+---------------+---------------+



In [18]:
# Top 10 by brand / product
spark.sql("""
    SELECT p.BRAND, p.product_name, SUM(f.sales_amount) AS total_sales
    FROM fact_sales f JOIN dim_product p ON f.product_id = p.product_id
    GROUP BY p.BRAND, p.product_name
    ORDER BY total_sales DESC
""").show(10)


+-----+--------------------+-------------+
|BRAND|        product_name|  total_sales|
+-----+--------------------+-------------+
|SPECS|BARRICADA ULTIMA ...| 2.19249485E9|
|SPECS|ACCELERATOR LIGHT...|  2.0175234E9|
|SPECS|ACCELERATOR LIGHT...|  1.8026848E9|
|SPECS|BARRICADA ULTIMA ...| 1.78032677E9|
|SPECS|ACCELERATOR LIGHT...|1.668244736E9|
|SPECS|BARRICADA ULTRA I...|  1.6584126E9|
|SPECS|BARRICADA ULTIMA ...|1.648265329E9|
|SPECS|BARRICADA ULTRA I...|  1.6314246E9|
|SPECS| BRAVE - BLUE/ORANGE|  1.5589074E9|
|SPECS|BRAVE - BLACK/ELE...|  1.5457128E9|
+-----+--------------------+-------------+
only showing top 10 rows


In [19]:
# Monthly trend by salesperson
spark.sql("""
    SELECT s.salesman_name, date_format(f.invoice_date, 'yyyy-MM') AS month, SUM(f.sales_amount) AS monthly_sales
    FROM fact_sales f JOIN dim_salesman s ON f.salesman_id = s.salesman_id
    GROUP BY s.salesman_name, month
    ORDER BY month DESC, monthly_sales DESC
""").show(10)


+--------------------+-------+-------------+
|       salesman_name|  month|monthly_sales|
+--------------------+-------+-------------+
|        Eddy Handoko|2017-02|  8.1575982E9|
|       Hendra Irawan|2017-02|   7.981302E9|
|       Sjilvia Yusuf|2017-02|   6.840765E9|
|      Hamal Abdullah|2017-02|6.462241784E9|
|       Management HQ|2017-02|6.234877711E9|
|       Agussyahbandi|2017-02|   4.010462E9|
|Siswanto Edie Pra...|2017-02|  3.7753508E9|
|      Theo Kurniawan|2017-02|  3.4423208E9|
|     Yangki Fiktoria|2017-02|  2.6565434E9|
| Eko Febrianto Piero|2017-02|   6.428242E8|
+--------------------+-------+-------------+
only showing top 10 rows


In [20]:
# Monthly trend by territory
spark.sql("""
    SELECT t.territory_name, date_format(f.invoice_date, 'yyyy-MM') AS month, SUM(f.sales_amount) AS monthly_sales
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_id = c.customer_id
    JOIN dim_territory t ON c.territory_id = t.territory_id
    GROUP BY t.territory_name, month
    ORDER BY month DESC, monthly_sales DESC
""").show(10)


+-------------------+-------+-------------+
|     territory_name|  month|monthly_sales|
+-------------------+-------+-------------+
|          Jabotabek|2017-02|1.52860678E10|
|        Jawa Tengah|2017-02|   8.782819E9|
|            Sumatra|2017-02|   8.757322E9|
|         Jawa Barat|2017-02|  4.0803768E9|
|         Kalimantan|2017-02|  3.6816882E9|
|           Sulawesi|2017-02|3.235284384E9|
|         Jawa Timur|2017-02|   2.357223E9|
|      Bali & Lombok|2017-02|   6.885464E8|
|Papua/Corporate/Mgt|2017-02|   1.856944E8|
|        Teritory 13|2017-02|    1.67809E8|
+-------------------+-------+-------------+
only showing top 10 rows


In [21]:
# Monthly trend by product category and by city
spark.sql("""
    SELECT p.CATEGORY, date_format(f.invoice_date, 'yyyy-MM') AS month, SUM(f.sales_amount) AS monthly_sales
    FROM fact_sales f JOIN dim_product p ON f.product_id = p.product_id
    GROUP BY p.CATEGORY, month
    ORDER BY month DESC, monthly_sales DESC
""").show(10)

spark.sql("""
    SELECT c.city, date_format(f.invoice_date, 'yyyy-MM') AS month, SUM(f.sales_amount) AS monthly_sales
    FROM fact_sales f JOIN dim_customer c ON f.customer_id = c.customer_id
    GROUP BY c.city, month
    ORDER BY month DESC, monthly_sales DESC
""").show(10)


+--------+-------+---------------+
|CATEGORY|  month|  monthly_sales|
+--------+-------+---------------+
|     SFW|2017-02|4.4114045691E10|
|     PFW|2017-02|  3.730591134E9|
|     SAH|2017-02|   3.55151103E9|
|     MFW|2017-02|   1.22117284E9|
|     TAH|2017-02|     1.186658E8|
|     PAH|2017-02|     1.009048E8|
|     MAH|2017-02|       1.8924E7|
|     OAH|2017-02|      2458400.0|
|     SFW|2017-01|3.6645861499E10|
|     PFW|2017-01|  4.166592614E9|
+--------+-------+---------------+
only showing top 10 rows
+----------+-------+-------------+
|      city|  month|monthly_sales|
+----------+-------+-------------+
|   Jakarta|2017-02|1.03114734E10|
| Tangerang|2017-02|   4.936535E9|
|   Bandung|2017-02|  3.7172416E9|
|      Solo|2017-02|   3.708293E9|
|    Bekasi|2017-02|  2.6895286E9|
|    Kendal|2017-02|  2.1549412E9|
|Yogyakarta|2017-02|  1.8726436E9|
|   Makasar|2017-02|1.736476984E9|
|     Depok|2017-02|   1.735152E9|
|     Medan|2017-02|   1.695788E9|
+----------+-------+----------

### Product profitability and segmentation

Two analysis queries used for the profitability scatter and the
sales-vs-quantity quadrant chart in the deck.

In [22]:
spark.sql("""
    WITH product_profit AS (
        SELECT
            dp.product_id, dp.product_name, dp.BRAND, dp.CATEGORY,
            SUM(fs.quantity) AS total_quantity,
            SUM(fs.sales_amount) AS total_sales,
            SUM(fs.cost) AS total_cost,
            SUM(fs.sales_amount) - SUM(fs.cost) AS total_profit
        FROM fact_sales fs
        LEFT JOIN dim_product dp ON fs.product_id = dp.product_id
        GROUP BY dp.product_id, dp.product_name, dp.BRAND, dp.CATEGORY
    )
    SELECT *, ROUND(total_profit::numeric * 100 / NULLIF(total_sales::numeric, 0), 2) AS profit_margin_pct
    FROM product_profit
    ORDER BY total_profit DESC
""").show(10)


+----------+--------------------+-----+--------+--------------+------------+--------------------+-------------------+-----------------+
|product_id|        product_name|BRAND|CATEGORY|total_quantity| total_sales|          total_cost|       total_profit|profit_margin_pct|
+----------+--------------------+-----+--------+--------------+------------+--------------------+-------------------+-----------------+
| 902746-NS|OPTIMUS SOCKS - B...|SPECS|     SAH|       18498.0|7.35976524E8|       3.144652286E8|      4.215112954E8|            57.27|
| 100700-41|ACCELERATOR LIGHT...|SPECS|     SFW|         661.0|  4.625678E8|        1.49430287E8|       3.13137513E8|            67.70|
| 100700-40|ACCELERATOR LIGHT...|SPECS|     SFW|         642.0|  4.492716E8|        1.45135014E8|       3.04136586E8|            67.70|
| 400563-41|BARRICADA ULTIMA ...|SPECS|     SFW|        1197.0|  5.126422E8|       2.139942523E8|      2.986479477E8|            58.26|
| 100702-41|ACCELERATOR LIGHT...|SPECS|     SFW|

## 7. Data Mart (CSV)

Aggregated, BI-tool-ready exports. Consolidated to the tables actually used
downstream — the source notebook exported a non-seasonal `inventory_summary`
and `aging_summary` alongside seasonal `_v2` versions of the same data;
kept only the seasonal ones since they're a strict superset.

In [23]:
def export_csv(df, name):
    df.write.mode("overwrite").option("header", True).csv(f"{PATHS['datamart']}/{name}")
    print(f"{name} exported.")


sales_summary = (
    fact_sales.groupBy("invoice_date")
    .agg(F.sum("quantity").alias("total_qty"), F.sum("sales_amount").alias("total_sales"))
    .orderBy("invoice_date")
)
export_csv(sales_summary, "sales_summary")

inventory_summary = (
    daily_balance.groupBy("date", "season")
    .agg(F.sum("closing_qty").alias("stock_qty"), F.sum("closing_value").alias("stock_value"))
    .orderBy("date", "season")
)
export_csv(inventory_summary, "inventory_summary_by_season")

aging_summary = (
    aging_fact.groupBy("season", "age_bucket")
    .agg(F.sum("total_qty").alias("stock_qty"), F.sum("total_value").alias("stock_value"))
    .orderBy("season", "age_bucket")
)
export_csv(aging_summary, "aging_summary_by_season")

log_step("datamart_export", "SUCCESS", None)


sales_summary exported.
inventory_summary_by_season exported.
aging_summary_by_season exported.


## 8. Data Masking

Customer names and phone numbers are masked before this dimension is shared
outside the data team.

In [24]:
masked_customer = (
    gold_tables["dim_customer"]
    .withColumn("customer_name", F.regexp_replace("customer_name", r"(?<=.{2}).", "*"))
)

masked_customer.select("customer_id", "customer_name").show(10, False)

masked_customer.write.mode("overwrite").parquet(f"{PATHS['gold']}/dim_customer_masked")
log_step("data_masking", "SUCCESS", masked_customer.count())


+-----------+-------------------------------+
|customer_id|customer_name                  |
+-----------+-------------------------------+
|1010.0     |ED**********************       |
|10100.0    |EK*********************        |
|101000.0   |EK***************              |
|1010000.0  |ES*****************************|
|101A001    |AA***********                  |
|101A002    |AB*******                      |
|101A003    |AD*********                    |
|101A004    |AG************************     |
|101A005    |AK****************             |
|101A006    |AN***********************      |
+-----------+-------------------------------+
only showing top 10 rows


## Pipeline run log

In [25]:
spark.read.option("header", True).csv(LOG_FILE).show(truncate=False)


+--------------------------+----------------------------+-------+------+
|timestamp                 |step                        |status |rows  |
+--------------------------+----------------------------+-------+------+
|2026-07-13 13:22:12.429278|fact_sales                  |SUCCESS|58816 |
|2026-07-13 13:22:12.955556|fact_inventory              |SUCCESS|11766 |
|2026-08-24 14:53:07.593143|bronze_ingestion            |SUCCESS|128554|
|2026-08-24 14:55:20.185172|silver_layer                |SUCCESS|124145|
|2026-08-24 14:57:03.159238|gold_dimensions             |SUCCESS|39209 |
|2026-08-24 14:57:12.893445|fact_sales                  |SUCCESS|58816 |
|2026-08-24 14:57:21.366151|fact_inventory_daily_balance|SUCCESS|11766 |
|2026-08-24 14:57:34.410519|fact_inventory_aging        |SUCCESS|1342  |
|2026-08-24 14:58:12.530524|datamart_export             |SUCCESS|NULL  |
|2026-08-24 14:58:18.305659|data_masking                |SUCCESS|1138  |
+--------------------------+-----------------------